# Ejemplo 2 — Instruction fine-tuning para *sentiment analysis*

Este notebook ilustra el **instruction fine-tuning**:
el modelo se entrena con **instrucciones en lenguaje natural**.

## Objetivo
Ajustar un modelo tipo T5/FLAN-T5 para que responda a instrucciones del tipo:

> `Classify the sentiment of this review: ...`

## Idea clave
- **Entrada**: instrucción + texto.
- **Salida**: texto (`positive` / `negative`).
- El modelo no aprende solo una etiqueta; aprende a **seguir instrucciones**.

## Qué diferencia este notebook
Frente al fine-tuning clásico:

- **Fine-tuning clásico**:  
  `I love this movie` → `positive`

- **Instruction fine-tuning**:  
  `Classify the sentiment: I love this movie` → `positive`

La tarea se expresa explícitamente como una instrucción.

In [ ]:
# Si hace falta, descomenta:
# !pip install -q transformers datasets evaluate accelerate sentencepiece

import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import evaluate

In [ ]:
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
dataset = load_dataset("glue", "sst2")

small_train = dataset["train"].shuffle(seed=42).select(range(800))
small_valid = dataset["validation"].shuffle(seed=42).select(range(200))

label_names = {0: "negative", 1: "positive"}

def preprocess(batch):
    inputs = [f"Classify the sentiment of this sentence: {x}" for x in batch["sentence"]]
    targets = [label_names[y] for y in batch["label"]]

    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(text_target=targets, max_length=8, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = small_train.map(preprocess, batched=True, remove_columns=small_train.column_names)
valid_ds = small_valid.map(preprocess, batched=True, remove_columns=small_valid.column_names)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_name)

accuracy = evaluate.load("accuracy")

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip().lower() for p in decoded_preds]
    decoded_labels = [l.strip().lower() for l in decoded_labels]

    preds = [1 if p == "positive" else 0 for p in decoded_preds]
    refs = [1 if l == "positive" else 0 for l in decoded_labels]

    return {"accuracy": accuracy.compute(predictions=preds, references=refs)["accuracy"]}

training_args = Seq2SeqTrainingArguments(
    output_dir="./ift-sentiment-flan-t5",
    learning_rate=5e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=8,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## Entrenamiento

Aquí el modelo se ajusta con ejemplos de **instrucción + respuesta**.

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

## Inferencia con nuevas instrucciones

In [ ]:
examples = [
    "Classify the sentiment of this sentence: I loved the soundtrack and the acting.",
    "Classify the sentiment of this sentence: The plot was weak and the ending was terrible.",
    "Classify the sentiment of this sentence: The movie started well but became boring.",
]

inputs = tokenizer(examples, return_tensors="pt", padding=True, truncation=True)
generated = model.generate(**inputs, max_new_tokens=4)
preds = tokenizer.batch_decode(generated, skip_special_tokens=True)

for x, p in zip(examples, preds):
    print(f"{x}\n -> {p}\n")

## Conclusión

Este notebook representa **instruction fine-tuning**:

- el modelo recibe una **instrucción en lenguaje natural**;
- la salida también es **texto**;
- el objetivo es que aprenda a **seguir instrucciones**, no solo a clasificar.